In [1]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


PyTorch version: 2.5.1+cu121
CUDA available: True
Device: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
import torch
import torch.nn as nn
import torchvision.models as models

In [3]:
class ResNetMLP(nn.Module):
    def __init__(self, num_classes):
        super(ResNetMLP, self).__init__()
        
        # 1. Load a pretrained ResNet (e.g. ResNet18)
        base_model = models.resnet18(pretrained=True)
        
        # 2. Remove the final FC layer -> get features instead
        self.feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
        
        # 3. MLP classifier head
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),   # ResNet18 outputs 512-dim features
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.feature_extractor(x)   # shape: (batch, 512, 1, 1)
        features = features.view(features.size(0), -1)  # flatten
        out = self.classifier(features)
        return out
